# 02 · Easy · Choose what to scan

Change *what* is scanned without editing files: the policy below is written inline.
In practice you would put the same YAML in your `policy.yaml`.

- **scope** decides what is read at all.
- **exclusions** are read but not analyzed. They still appear as `excluded`.

In [ ]:
%pip install /Volumes/main/default/lakehouse_gc/lakehouse_gc-0.2.2-py3-none-any.whl

In [ ]:
from lakehouse_gc import run
from lakehouse_gc.core import loads_policy

POLICY_YAML = """
policy_version: 1

scope:
  include_catalogs: ["main"]          # empty list = all catalogs
  include_schemas: ["main.sales"]     # catalog.schema
  exclude_tables: ["main.sales.tmp_*"]

exclusions:
  - resource_type: table              # uc types: catalog, schema, table
    pattern: "main.sales.legacy_*"
    reason: "Migration hold"
    expires_on: 2026-12-31            # applies through this date; null = permanent

rules:
  uc:
    stale_table_days: 90
    min_table_size_gb: 1
    required_tags: ["cost_center"]
"""

policy = loads_policy(POLICY_YAML)
result = run(policy=policy, scanners=["uc"], spark=spark)
print(result.summary())

In [ ]:
print("Excluded (still visible):", [f.resource.resource_id for f in result.excluded_findings])
display(result.to_dataframe(spark))